# Groq EDA Executive Report — Ready-to-Run Notebook

Upload a CSV → run automated EDA → generate five insights → create an approximately 200-word executive summary → download a PowerPoint with a title slide, five insight slides, and a recommendation slide.

**No manual package installation needed:** the first code cell installs required packages automatically in the notebook environment. Groq is optional; without an API key, a local rule-based summary is generated so the full workflow still runs.


In [ ]:
%pip -q install pandas matplotlib python-pptx groq
import os, json, textwrap, getpass, warnings
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN
from pptx.dml.color import RGBColor
warnings.filterwarnings('ignore')
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
print('Setup complete. Upload your CSV in the next cell.')


In [ ]:
# In Google Colab, choose your CSV file when prompted. Outside Colab, enter a file path.
try:
    from google.colab import files
    uploaded = files.upload()
    csv_path = next(iter(uploaded))
except Exception:
    csv_path = input('Enter path to CSV file (or use data/sample_sales.csv): ').strip() or 'data/sample_sales.csv'
df = pd.read_csv(csv_path)
print(f'Loaded {len(df):,} rows × {len(df.columns)} columns')
display(df.head())


In [ ]:
# Automated EDA: structure, missing values, duplicates, numeric summary, and useful charts.
eda = {
    'rows': int(df.shape[0]), 'columns': int(df.shape[1]),
    'duplicate_rows': int(df.duplicated().sum()),
    'missing_by_column': df.isna().sum().astype(int).to_dict(),
    'numeric_summary': df.describe(include='number').round(2).to_dict(),
    'dtypes': {c: str(t) for c, t in df.dtypes.items()}
}
print('DATASET PROFILE')
print(f"Rows: {eda['rows']:,} | Columns: {eda['columns']} | Duplicate rows: {eda['duplicate_rows']}")
display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'missing_%': (df.isna().mean()*100).round(1)}))
display(df.describe(include='all').T)
numeric = df.select_dtypes(include='number')
if not numeric.empty:
    ax = numeric.hist(figsize=(12, 7), bins=15)
    plt.tight_layout(); plt.savefig(OUT/'numeric_distributions.png', dpi=150, bbox_inches='tight'); plt.show()
eda_path = OUT/'eda_profile.json'
eda_path.write_text(json.dumps(eda, indent=2, default=str), encoding='utf-8')
print('EDA profile saved to', eda_path)


In [ ]:
# Generate five data insights from the uploaded CSV. Uses numeric columns and categorical groupings when available.
insights = []
insights.append(f'The dataset contains {len(df):,} rows and {len(df.columns)} columns, providing the current analysis base.')
missing_total = int(df.isna().sum().sum())
insights.append(f'Data completeness: {missing_total:,} missing cells across the dataset ({missing_total / max(df.size,1) * 100:.1f}% of all cells).')
insights.append(f'Duplicate records: {int(df.duplicated().sum()):,} rows are exact duplicates and may need review before reporting.')
if not numeric.empty:
    for col in numeric.columns:
        vals = numeric[col].dropna()
        if len(vals) >= 2:
            insights.append(f'{col}: average {vals.mean():,.2f}; minimum {vals.min():,.2f}; maximum {vals.max():,.2f}.')
        if len(insights) >= 5: break
if len(insights) < 5:
    for col in df.columns:
        if col not in numeric.columns:
            insights.append(f'{col}: {df[col].nunique(dropna=True):,} distinct non-null values across {df[col].notna().sum():,} populated rows.')
        if len(insights) >= 5: break
insights = insights[:5]
while len(insights) < 5:
    insights.append('Review column definitions and validate data quality before making business decisions.')
print('FIVE KEY INSIGHTS')
for i, insight in enumerate(insights, 1): print(f'{i}. {insight}')


In [ ]:
# Optional Groq-powered executive summary. Leave GROQ_API_KEY blank to use the built-in fallback.
groq_key = os.getenv('GROQ_API_KEY', '').strip()
if not groq_key:
    try:
        groq_key = getpass.getpass('Optional: enter Groq API key (press Enter to skip): ').strip()
    except Exception:
        groq_key = ''
summary = ''
if groq_key:
    try:
        from groq import Groq
        client = Groq(api_key=groq_key)
        response = client.chat.completions.create(
            model='llama-3.3-70b-versatile',
            messages=[
                {'role':'system','content':'You are a business data analyst. Write a clear executive summary of about 200 words. Use only the supplied evidence, distinguish facts from hypotheses, and end with practical next steps.'},
                {'role':'user','content':'Dataset profile: '+json.dumps({'rows':len(df),'columns':list(df.columns),'missing_cells':int(df.isna().sum().sum()),'duplicate_rows':int(df.duplicated().sum())})+'\nFive insights:\n'+'\n'.join(f'{i+1}. {x}' for i,x in enumerate(insights))+'\nWrite the executive summary.'}
            ], temperature=0.2, max_tokens=450)
        summary = response.choices[0].message.content.strip()
        summary_source = 'Groq AI'
    except Exception as e:
        print('Groq request failed; using local fallback instead. Details:', str(e)[:250])
if not summary:
    summary = ('This analysis reviews a dataset containing '+f'{len(df):,} rows and {len(df.columns)} columns. '+
               f'The initial quality check identified {int(df.isna().sum().sum()):,} missing cells and {int(df.duplicated().sum()):,} exact duplicate rows. '+
               'The key findings below summarize the dataset profile and available numeric measures. '+
               'Before using these results for operational or strategic decisions, confirm the meaning and units of each field, investigate missing or duplicate records, and validate unusual values with the data owner. '+
               'Where the data supports comparisons, compare performance across relevant groups and time periods rather than relying only on overall averages. '+
               'Prioritize the measures most closely tied to business outcomes, then assign an owner and review date for each follow-up action. '+
               'These results describe the supplied file only and should not be treated as proof of causation. Further analysis with business context and validated source data will help stakeholders determine which actions are most likely to improve performance.')
    summary_source = 'Built-in fallback (no Groq API key used)'
print('Summary source:', summary_source)
print('\nEXECUTIVE SUMMARY\n')
print(summary)


In [ ]:
# Build PowerPoint: title slide + five insight slides + recommendation slide.
prs = Presentation()
prs.slide_width = Inches(13.333); prs.slide_height = Inches(7.5)
def add_title(slide, title, subtitle=None):
    box = slide.shapes.add_textbox(Inches(0.7), Inches(0.35), Inches(12), Inches(0.75))
    p = box.text_frame.paragraphs[0]; p.text = title; p.font.size = Pt(28); p.font.bold = True; p.font.color.rgb = RGBColor(31, 78, 121)
    if subtitle:
        b = slide.shapes.add_textbox(Inches(0.75), Inches(1.15), Inches(11.8), Inches(0.45))
        q = b.text_frame.paragraphs[0]; q.text = subtitle; q.font.size = Pt(13); q.font.color.rgb = RGBColor(90, 90, 90)
def add_body(slide, text, top=1.8, size=22):
    box = slide.shapes.add_textbox(Inches(0.9), Inches(top), Inches(11.5), Inches(4.8))
    tf = box.text_frame; tf.word_wrap = True
    p = tf.paragraphs[0]; p.text = text; p.font.size = Pt(size); p.font.color.rgb = RGBColor(45, 45, 45)
slide = prs.slides.add_slide(prs.slide_layouts[6]); add_title(slide, 'Executive Data Analysis Report', 'Automated EDA • AI-assisted narrative • Generated from uploaded CSV')
add_body(slide, f'Dataset: {Path(csv_path).name}\nRows: {len(df):,}\nColumns: {len(df.columns)}\nSummary source: {summary_source}', top=2.1, size=24)
for i, insight in enumerate(insights, 1):
    slide = prs.slides.add_slide(prs.slide_layouts[6]); add_title(slide, f'Key Insight {i}', f'Finding {i} of 5')
    add_body(slide, insight, top=2.0, size=25)
slide = prs.slides.add_slide(prs.slide_layouts[6]); add_title(slide, 'Recommendations & Next Steps', 'Prioritize validation, then turn findings into action')
recommendations = [
    'Validate column definitions, units, and source-system accuracy with the data owner.',
    'Investigate missing values and duplicate records; document any cleaning decisions.',
    'Compare key metrics across relevant categories or time periods where the data permits.',
    'Assign owners, success measures, and review dates to the highest-priority actions.',
    'Refresh the analysis when new data arrives and track whether outcomes improve.'
]
add_body(slide, '\n'.join('• '+x for x in recommendations), top=1.65, size=19)
ppt_path = OUT/'Groq_EDA_Executive_Report.pptx'
prs.save(ppt_path)
summary_path = OUT/'executive_summary.txt'; summary_path.write_text(summary, encoding='utf-8')
insights_path = OUT/'five_insights.txt'; insights_path.write_text('\n'.join(f'{i+1}. {x}' for i,x in enumerate(insights)), encoding='utf-8')
print('Created:', ppt_path.resolve())
print('Created:', summary_path.resolve())
print('Created:', insights_path.resolve())


In [ ]:
# In Google Colab, download the finished PowerPoint and supporting outputs.
try:
    from google.colab import files
    files.download(str(ppt_path))
except Exception:
    print('Open the outputs folder to access the generated files:', OUT.resolve())


## Notes
- The notebook automatically installs its Python dependencies in the notebook runtime.
- To use Groq AI, provide a Groq API key when prompted. If skipped or unavailable, the notebook still completes using its built-in fallback summary.
- The five insight slides are generated from the uploaded dataset. Check the findings before presenting them to stakeholders.
